# Leaflet cluster map of talk locations

This notebook generates an interactive map of all your talk locations.

**Prerequisites:** Make sure you have the required packages installed:
```bash
poetry install
```

Or if not using poetry:
```bash
pip install python-frontmatter getorg geopy pyyaml
```

**Note:** You can also run `python talkmap.py` from the command line as a standalone script.


In [ ]:
# Import required libraries
import glob
import os
import yaml
from pathlib import Path
from geopy import Nominatim
from geopy.exc import GeocoderTimedOut
import getorg


In [ ]:
# Function to parse YAML frontmatter from markdown files
def parse_frontmatter(file_path):
    """
    Parse YAML frontmatter from a markdown file.
    Returns a dict of frontmatter data or None if parsing fails.
    """
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            content = f.read()
        
        # Check if file starts with ---
        if not content.startswith('---'):
            return None
        
        # Split on second ---
        parts = content.split('---', 2)
        if len(parts) < 3:
            return None
        
        # Parse YAML
        try:
            data = yaml.safe_load(parts[1])
            return data if isinstance(data, dict) else None
        except yaml.YAMLError:
            return None
    except Exception as e:
        print(f"Error reading {file_path}: {e}")
        return None

# Collect the Markdown files
print("Scanning _talks/ directory for markdown files...")
talk_files = glob.glob("_talks/*.md")
print(f"Found {len(talk_files)} talk files\n")


In [ ]:
# Set the default timeout, in seconds
TIMEOUT = 5

# Prepare to geolocate
print("Initializing geocoder...\n")
geocoder = Nominatim(user_agent="academicpages.github.io")
location_dict = {}
processed_count = 0
skipped_count = 0


## Processing and Geocoding

The following cell processes each talk file and geocodes its location. If you encounter timeouts, you can increase the TIMEOUT value above.


In [ ]:
# Perform geolocation
print("Processing talks and geocoding locations:\n")

for file_path in talk_files:
    # Parse the file
    data = parse_frontmatter(file_path)
    
    if data is None:
        print(f"⚠ Skipped {file_path}: Could not parse frontmatter")
        skipped_count += 1
        continue

    # Press on if the location is not present
    if 'location' not in data:
        print(f"⚠ Skipped {file_path}: No location field")
        skipped_count += 1
        continue

    # Prepare the description
    try:
        title = str(data.get('title', 'Unknown')).strip()
        venue = str(data.get('venue', 'Unknown venue')).strip()
        location = str(data['location']).strip()
        description = f"{title}<br />{venue}; {location}"

        # Geocode the location and report the status
        try:
            result = geocoder.geocode(location, timeout=TIMEOUT)
            location_dict[description] = result
            if result:
                print(f"✓ {description}")
                print(f"  → ({result.latitude}, {result.longitude})")
            else:
                print(f"✗ {description}")
                print(f"  → Could not geocode location")
            processed_count += 1
        except ValueError as ex:
            print(f"✗ Error: geocode failed on '{location}': {ex}")
        except GeocoderTimedOut as ex:
            print(f"✗ Error: geocode timed out on '{location}': {ex}")
        except Exception as ex:
            print(f"✗ Unexpected error processing '{location}': {ex}")
    except KeyError as ex:
        print(f"⚠ Skipped {file_path}: Missing required field {ex}")
        skipped_count += 1

print(f"\n{'='*60}")
print(f"Processed: {processed_count} talks")
print(f"Skipped: {skipped_count} talks")
print(f"Successfully geocoded: {sum(1 for v in location_dict.values() if v is not None)} locations")


In [ ]:
# Generate the map
if location_dict:
    print(f"\nGenerating map output...")
    try:
        m = getorg.orgmap.create_map_obj()
        getorg.orgmap.output_html_cluster_map(
            location_dict, 
            folder_name="talkmap", 
            hashed_usernames=False
        )
        print("✓ Map saved to talkmap/ directory")
    except Exception as ex:
        print(f"Error generating map: {ex}")
else:
    print("No locations found to map")
